# PlantVillage: Tiếp Tục Huấn Luyện (Resume / Continuous Training) Mô Hình VGG16 Tốt Nhất

**Dự án:** Phân loại bệnh hại trên lá cây trồng (Plant Disease Classification - 38 lớp)  
**Mục tiêu Notebook:**
1. Tự động tìm và nạp checkpoint mô hình VGG16 tốt nhất hiện có (`models/vgg16_model.keras` hoặc `outputs/vgg16/checkpoints/vgg16_finetuned.keras`).
2. Kiểm tra cấu trúc mô hình, trạng thái đóng băng/mở khóa các tầng (đặc biệt là Block 5 và đầu phân loại chung GAP + Dropout 0.3 + Dense 38).
3. Nạp pipeline dữ liệu chuẩn theo sơ đồ nhóm (Stratified Split 70/15/15, Augmentation, tính trọng số lớp class weights, VGG16 preprocess BGR zero-centering).
4. Thiết lập cơ chế **EarlyStopping** (`patience=5`, `restore_best_weights=True`) và **ReduceLROnPlateau** (`patience=2`) để **tiếp tục huấn luyện tự động cho đến khi hội tụ hoàn toàn (khi nào không cải thiện nữa thì tự dừng)**.
5. Đánh giá kiểm thử độc lập mô hình sau khi train tiếp trên toàn bộ 8.143 ảnh của tập Test Set và so sánh sự cải thiện.
6. Lưu lại checkpoint mới nhất và cập nhật model tốt nhất vào `models/vgg16_model.keras`.


## 1. Khởi Tạo Môi Trường & Thiết Lập Cấu Hình

Khai báo các thư viện, cố định Random Seed (425), cấu hình môi trường tính toán và định vị thư mục gốc dự án.


In [ ]:
import os
import sys
import json
import time
from pathlib import Path

# Tránh xung đột OpenMP trên Windows và tắt log thừa của TensorFlow
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'

# Định vị thư mục gốc của dự án
NOTEBOOK_DIR = Path('.').resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == '_ipynb' else NOTEBOOK_DIR
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

from src.utils.reproducibility import set_seed, get_environment_info
from src.utils.config import load_config, BASE_DIR
from src.data.dataset import (
    load_dataset_splits,
    compute_class_imbalance_weights,
    create_tf_dataset,
)
from src.training.callbacks import create_experiment_callbacks
from src.evaluation.evaluate import evaluate_model_on_test_set

# Thiết lập Seed đảm bảo tính tái lập
SEED = 425
set_seed(SEED)

env_info = get_environment_info()
print('=' * 65)
print('[OK] Môi trường tính toán:')
print(f'  - Python    : {sys.version.split()[0]}')
print(f'  - TensorFlow: {tf.__version__}')
print(f'  - Keras     : {keras.__version__}')
gpu_msg = env_info['gpu_devices'] if env_info['gpu_available'] else 'Chạy trên CPU'
print(f'  - GPU/CUDA  : {gpu_msg}')
print('=' * 65)


## 2. Nạp Tập Dữ Liệu & Pipeline Tiền Xử Lý (Data Pipeline)

Nạp thông tin phân chia tập train (70%), val (15%), test (15%), tính toán class weights để xử lý mất cân bằng lớp và khởi tạo `tf.data.Dataset` tối ưu có tăng cường dữ liệu cho tập Train.


In [ ]:
cfg = load_config()
paths = cfg['resolved_paths']

# Nạp tập dữ liệu đã chia phân tầng
train_df, val_df, test_df, idx_to_class, class_to_idx = load_dataset_splits(
    csv_dir=paths['csv_dir'],
    reports_dir=paths['report_dir']
)

print(f'[+] Tập huấn luyện (Train): {len(train_df):,} ảnh (70%)')
print(f'[+] Tập kiểm định (Val)   : {len(val_df):,} ảnh (15%)')
print(f'[+] Tập kiểm thử (Test)   : {len(test_df):,} ảnh (15%)')
print(f'[+] Tổng số lớp           : {len(idx_to_class)} lớp')

# Tính trọng số các lớp (Class Weights) để giải quyết mất cân bằng số lượng ảnh
class_weights = compute_class_imbalance_weights(
    train_df=train_df,
    val_df=val_df,
    test_df=test_df,
    reports_dir=paths['report_dir'],
    num_classes=len(idx_to_class)
)

BATCH_SIZE = cfg['dataset'].get('batch_size', 32)
IMG_SIZE = tuple(cfg['dataset'].get('img_size', [224, 224]))

# Tạo tf.data pipeline: Train có Data Augmentation; Val/Test giữ nguyên ảnh gốc
train_dataset = create_tf_dataset(
    train_df,
    is_training=True,
    use_augmentation=True,
    aug_config=cfg.get('augmentation'),
    batch_size=BATCH_SIZE,
    img_size=IMG_SIZE,
    seed=SEED,
)

val_dataset = create_tf_dataset(
    val_df,
    is_training=False,
    use_augmentation=False,
    batch_size=BATCH_SIZE,
    img_size=IMG_SIZE,
    seed=SEED,
)

test_dataset = create_tf_dataset(
    test_df,
    is_training=False,
    use_augmentation=False,
    batch_size=BATCH_SIZE,
    img_size=IMG_SIZE,
    seed=SEED,
)

print('[OK] Đã chuẩn bị xong toàn bộ tf.data pipeline.')


## 3. Tự Động Tìm & Nạp Checkpoint VGG16 Tốt Nhất Hiện Có

Quét các thư mục checkpoint và nạp tệp mô hình tối ưu nhất (`vgg16_finetuned.keras` hoặc `models/vgg16_model.keras`).


In [ ]:
candidate_checkpoints = [
    PROJECT_ROOT / 'outputs/vgg16/checkpoints/vgg16_finetuned.keras',
    PROJECT_ROOT / 'models/vgg16_model.keras',
    PROJECT_ROOT / 'outputs/vgg16/checkpoints/vgg16_feature_extraction.keras',
]

chosen_checkpoint = None
for cp in candidate_checkpoints:
    if cp.exists():
        chosen_checkpoint = cp
        break

if chosen_checkpoint is None:
    raise FileNotFoundError('Không tìm thấy checkpoint VGG16 nào để tiếp tục huấn luyện!')

size_mb = chosen_checkpoint.stat().st_size / (1024 * 1024)
print('=' * 65)
print(f'[+] NẠP MÔ HÌNH TỐT NHẤT TỪ: {chosen_checkpoint}')
print(f'    Dung lượng tệp: {size_mb:.1f} MB')
print('=' * 65)

model = keras.models.load_model(str(chosen_checkpoint))
print('[OK] Nạp mô hình thành công!')

# Đếm số lượng tham số
total_params = model.count_params()
trainable_params = sum(np.prod(p.shape) for p in model.trainable_weights)
non_trainable_params = total_params - trainable_params

print(f'  - Tổng tham số (Total Params)       : {total_params:,}')
print(f'  - Tham số huấn luyện (Trainable)   : {trainable_params:,}')
print(f'  - Tham số cố định (Non-trainable)  : {non_trainable_params:,}')


## 4. Kiểm Tra & Cấu Hình Mở Khóa Tầng (Trainable Layers)

Theo phương pháp luận của nhóm, giai đoạn Fine-tuning sẽ mở khóa khối cuối cùng (**Block 5**) của VGG16 (`block5_conv1`, `block5_conv2`, `block5_conv3`) và toàn bộ đầu phân loại (Head). Các khối trích xuất đặc trưng cơ bản (Block 1 - 4) được giữ đóng băng.


In [ ]:
from src.models.vgg16_model import set_backbone_trainable

# Lấy backbone VGG16 bên trong mô hình
base_model = None
for layer in model.layers:
    if 'vgg16' in layer.name.lower():
        base_model = layer
        break

if base_model is not None:
    ft_info = set_backbone_trainable(model, base_model, unfreeze_from_layer='block5_conv1')
    print('=' * 65)
    print('CẤU HÌNH MỞ KHÓA TẦNG CHO FINE-TUNING TIẾP TỤC:')
    print(f' - Các tầng được mở khóa: {ft_info["unfrozen_layers"]}')
    print(f' - Tham số Trainable     : {ft_info["trainable_params"]:,}')
    print(f' - Tham số Non-trainable : {ft_info["non_trainable_params"]:,}')
    print('=' * 65)
else:
    print('[i] Không tìm thấy layer con vgg16, giữ nguyên cấu hình trainable của mô hình.')


## 5. Thiết Lập Callbacks Dừng Tự Động (Stop-When-Done Engine)

Để mô hình **huấn luyện đến khi nào dừng thì thôi**:
- **`EarlyStopping`**: Theo dõi `val_loss`. Nếu sau **5 epochs liên tiếp** không giảm thêm (loss không cải thiện), quá trình train sẽ tự động dừng lại ngay lập tức và khôi phục lại bộ trọng số tốt nhất (`restore_best_weights=True`).
- **`ReduceLROnPlateau`**: Nếu sau **2 epochs** mà `val_loss` dậm chân tại chỗ, tự động giảm learning rate đi 5 lần (`factor=0.2`) để giúp mô hình len lỏi tìm cực tiểu tối ưu hơn.
- **`ModelCheckpoint`**: Luôn tự động ghi đè lưu lại checkpoint tốt nhất vào tệp `outputs/vgg16/checkpoints/vgg16_continued_best.keras`.
- **`CSVLogger`**: Lưu nhật ký từng epoch vào tệp CSV.


In [ ]:
from tensorflow.keras.callbacks import (
    ModelCheckpoint,
    EarlyStopping,
    ReduceLROnPlateau,
    CSVLogger,
)
from src.training.callbacks import TrainingTimerCallback

# Định cấu hình đường dẫn lưu kết quả tiếp tục huấn luyện
CONTINUED_CHECKPOINT = PROJECT_ROOT / 'outputs/vgg16/checkpoints/vgg16_continued_best.keras'
CONTINUED_HISTORY_CSV = PROJECT_ROOT / 'outputs/vgg16/histories/vgg16_continued_history.csv'

CONTINUED_CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
CONTINUED_HISTORY_CSV.parent.mkdir(parents=True, exist_ok=True)

# Thiết lập số epoch tối đa thật lớn (50 epochs), EarlyStopping sẽ tự ngắt khi hội tụ
MAX_EPOCHS = 50
PATIENCE_ES = 5  # Dừng sau 5 epochs không cải thiện
PATIENCE_LR = 2  # Giảm LR sau 2 epochs không cải thiện
INITIAL_LR = 1e-5  # Learning rate tối ưu cho Fine-Tuning

checkpoint_cb = ModelCheckpoint(
    filepath=str(CONTINUED_CHECKPOINT),
    monitor='val_loss',
    mode='min',
    save_best_only=True,
    verbose=1,
)

early_stopping_cb = EarlyStopping(
    monitor='val_loss',
    mode='min',
    patience=PATIENCE_ES,
    restore_best_weights=True,
    verbose=1,
)

reduce_lr_cb = ReduceLROnPlateau(
    monitor='val_loss',
    mode='min',
    factor=0.2,
    patience=PATIENCE_LR,
    min_lr=1e-8,
    verbose=1,
)

csv_logger_cb = CSVLogger(
    filename=str(CONTINUED_HISTORY_CSV),
    separator=',',
    append=False,
)

timer_cb = TrainingTimerCallback()

callbacks = [
    checkpoint_cb,
    early_stopping_cb,
    reduce_lr_cb,
    csv_logger_cb,
    timer_cb,
]

# Compile mô hình với Adam optimizer và learning rate nhỏ
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=INITIAL_LR),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    metrics=['accuracy'],
)

print('=' * 65)
print('[+] SẴN SÀNG TIẾP TỤC HUẤN LUYỆN:')
print(f'  - Max Epochs       : {MAX_EPOCHS}')
print(f'  - Initial LR       : {INITIAL_LR}')
print(f'  - Early Stopping   : Dừng nếu val_loss không giảm sau {PATIENCE_ES} epochs')
print(f'  - Reduce LR        : Giảm LR sau {PATIENCE_LR} epochs plateau')
print(f'  - Checkpoint Target: {CONTINUED_CHECKPOINT}')
print('=' * 65)


## 6. Thực Hiện Tiếp Tục Huấn Luyện (Training Loop)

Quá trình huấn luyện bắt đầu. Mô hình sẽ chạy liên tục qua các epochs cho đến khi thỏa mãn điều kiện EarlyStopping (hoặc đạt MAX_EPOCHS).


In [ ]:
print('\n>>> BẮT ĐẦU QUÁ TRÌNH TIẾP TỤC HUẤN LUYỆN ĐẾN KHI HỘI TỤ (STOP-WHEN-DONE)...\n')

history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=MAX_EPOCHS,
    callbacks=callbacks,
    class_weight=class_weights,
    verbose=1,
)

print('\n' + '=' * 65)
print('[+] QUÁ TRÌNH HUẤN LUYỆN ĐÃ KẾT THÚC THÀNH CÔNG!')
print(f'  - Tổng thời gian huấn luyện: {timer_cb.total_duration_seconds:.2f} giây ({timer_cb.total_duration_seconds/60:.2f} phút)')
print(f'  - Số epoch đã chạy thực tế : {len(history.history["loss"])}')
print('=' * 65)


## 7. Trực Quan Hóa Quá Trình Huấn Luyện (Loss & Accuracy Curves)

Vẽ đồ thị biểu diễn Loss và Accuracy của tập huấn luyện (Train) và tập kiểm định (Val) qua từng epoch tiếp theo.


In [ ]:
hist_df = pd.DataFrame(history.history)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Biểu đồ Accuracy
axes[0].plot(hist_df['accuracy'], label='Train Accuracy', color='#2b5c8f', linewidth=2, marker='o')
axes[0].plot(hist_df['val_accuracy'], label='Val Accuracy', color='#27ae60', linewidth=2, marker='s')
axes[0].set_title('Độ Chính Xác Qua Các Epoch (Continued Training)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch', fontsize=11)
axes[0].set_ylabel('Accuracy', fontsize=11)
axes[0].grid(True, linestyle='--', alpha=0.6)
axes[0].legend(fontsize=11)

# Biểu đồ Loss
axes[1].plot(hist_df['loss'], label='Train Loss', color='#c0392b', linewidth=2, marker='o')
axes[1].plot(hist_df['val_loss'], label='Val Loss', color='#e67e22', linewidth=2, marker='s')
axes[1].set_title('Hàm Mất Mát (Loss) Qua Các Epoch', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch', fontsize=11)
axes[1].set_ylabel('Loss', fontsize=11)
axes[1].grid(True, linestyle='--', alpha=0.6)
axes[1].legend(fontsize=11)

plt.tight_layout()
plot_path = PROJECT_ROOT / 'outputs/vgg16/reports/vgg16_continued_training_curves.png'
plot_path.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(plot_path, dpi=300)
plt.show()
print(f'[+] Đã lưu biểu đồ vào: {plot_path}')


## 8. Đánh Giá Toàn Diện Trên Tập Test Độc Lập (8.143 Ảnh)

Đo lường các chỉ số chính thức: Test Accuracy, Macro/Weighted Precision, Recall, F1-Score trên toàn bộ tập Test Set độc lập.


In [ ]:
# Nạp lại checkpoint có val_loss thấp nhất để đảm bảo kết quả tốt nhất
best_model = keras.models.load_model(str(CONTINUED_CHECKPOINT))

report_csv_path = PROJECT_ROOT / 'outputs/vgg16/reports/classification_report_vgg16_continued.csv'

eval_metrics = evaluate_model_on_test_set(
    model=best_model,
    test_dataset=test_dataset,
    test_df=test_df,
    idx_to_class=idx_to_class,
    report_csv_path=str(report_csv_path),
)

print('=' * 65)
print('KẾT QUẢ ĐÁNH GIÁ TRÊN TẬP KIỂM THỬ (TEST SET - 8,143 ẢNH):')
print(f'  - Test Accuracy   : {eval_metrics.get("accuracy", 0) * 100:.2f}%')
print(f'  - Test Loss       : {eval_metrics.get("loss", 0):.4f}')
print(f'  - Macro F1-Score  : {eval_metrics.get("macro_f1", 0):.4f}')
print(f'  - Weighted F1     : {eval_metrics.get("weighted_f1", 0):.4f}')
print(f'  - Macro Precision : {eval_metrics.get("macro_precision", 0):.4f}')
print(f'  - Macro Recall    : {eval_metrics.get("macro_recall", 0):.4f}')
print('=' * 65)


## 9. Cập Nhật Mô Hình Tốt Nhất Vào Thư Mục `models/`

Nếu kết quả Test Accuracy đạt yêu cầu, chúng ta tự động lưu hoặc ghi đè mô hình tối ưu vào `models/vgg16_model.keras` để các notebook kiểm thử đối chuẩn (05) sử dụng ngay.


In [ ]:
target_model_path = PROJECT_ROOT / 'models/vgg16_model.keras'

# Tạo bản sao lưu dự phòng cho mô hình cũ nếu cần
if target_model_path.exists():
    backup_path = PROJECT_ROOT / 'models/vgg16_model_backup.keras'
    import shutil
    shutil.copy2(target_model_path, backup_path)
    print(f'[+] Đã tạo bản sao lưu mô hình cũ tại: {backup_path}')

# Lưu mô hình mới vào models/
best_model.save(target_model_path)
print(f'[OK] Đã cập nhật mô hình VGG16 tốt nhất vào: {target_model_path}')
print(f'     Dung lượng tệp: {target_model_path.stat().st_size / (1024*1024):.1f} MB')
